# Project - Airline AI Assistant

AI Customer Support assistant for an Airline

In [75]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from google import genai
from google.genai import types
from PIL import Image


In [76]:
import base64
from io import BytesIO


In [77]:
# Initialization

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY2')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
    


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ


In [78]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

#gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url)
gemini_client = genai.Client(api_key=google_api_key)
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)


In [79]:
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-2.5-flash-lite"
ollama_model = "llama3.2:latest"

In [80]:
#Creating the DB
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [81]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [82]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [83]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [84]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [85]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [86]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

Tried Ollama to generate image but memory needed was too much for a normal PC so the experiment failed

In [ ]:
# using Ollama to generate image
# In terminal - ollama pull x/z-image-turbo 
# In terminal - pip install requests pillow  pip install openai pillow

In [ ]:
# Some imports for handling images

#import requests
# import base64
# from io import BytesIO
# from PIL import Image

In [ ]:
# def artist_ollama(city):
#     prompt = (
#         f"An image representing a vacation in {city}, "
#         f"showing tourist spots and everything unique about {city}, "
#         f"in a vibrant pop-art style."
#     )

#     response = ollama_client.images.generate(
#         model="x/z-image-turbo",
#         prompt=prompt,
#         size="1024x1024",
#         response_format="b64_json",
#     )

#     image_base64 = response.data[0].b64_json
#     image_data = base64.b64decode(image_base64)

#     return Image.open(BytesIO(image_data))

In [ ]:
# image_o = artist_ollama("New York City")
# display(image_o)

Tried using free tier gemini for image generation but did not work as free tier less for it, though code is good

In [ ]:
#gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


# def artist_gemini(city):
#     prompt = (
#         f"An image representing a {city}, "
#     )

#     response = gemini_client.models.generate_content(
#         model="gemini-2.5-flash-image",
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             response_modalities=["IMAGE", "TEXT"]
#         )
#     )

#     for candidate in response.candidates:
#         for part in candidate.content.parts:
#             if part.inline_data is not None:
#                 image_bytes = part.inline_data.data
#                 return Image.open(BytesIO(image_bytes))

#     raise ValueError("No image was returned by Gemini.")

In [ ]:
# img = artist_gemini("New York")
# img.show()

In [ ]:
# image_g = artist_gemini("New York City")
# display(image_g)

In [ ]:
#Open AI code - commented it so as to not to use it again and again
# def artist(city):
#     image_response = openai.images.generate(
#             model="dall-e-3",
#             prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
#             size="1024x1024",
#             n=1,
#             response_format="b64_json",
#         )
#     image_base64 = image_response.data[0].b64_json
#     image_data = base64.b64decode(image_base64)
#     return Image.open(BytesIO(image_data))

In [ ]:
# image = artist("New York City")
# display(image)

In [87]:
#added an image for multiple use in Gradio so that I dont have to use openai multiple times
img = Image.open("city_vacation.png")
#img

In [88]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [89]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = img #artist(cities[0])
    
    return history, voice, image


In [90]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [91]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo
